# Prompt Testing Sandbox

Scratchpad for testing prompts before committing them to experiments.
Use this to:
- Check how prompts tokenize
- See what the model actually predicts (before running full analysis)
- Test whether target tokens are single tokens
- Just talk to the model and see what it thinks

Nothing here needs to be clean or saved. This is the playground.

In [ ]:
import sys
sys.path.insert(0, '../..')

from src.models import load_model, unload

In [2]:
model, info = load_model("gpt2-medium")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-medium into HookedTransformer
Loaded gpt2-medium on mps
  24 layers | 16 heads | d_model=1024 | d_mlp=4096 | sequential attn+MLP


## Tokenization Check
Always verify your target is a single token before running a full experiment.

In [3]:
def check_token(model, text):
    """See how a string tokenizes. Use leading space for mid-sentence tokens."""
    tokens = model.to_tokens(text, prepend_bos=False)[0]
    print(f"'{text}' → {len(tokens)} token(s):")
    for i, t in enumerate(tokens):
        print(f"  [{i}] {t.item()} = '{model.to_string(t)}'")
    return tokens

# Examples — test your targets here
check_token(model, " Paris")
check_token(model, " Web")
check_token(model, " NVDA")
check_token(model, " screen")
check_token(model, " ARIA")

' Paris' → 1 token(s):
  [0] 6342 = ' Paris'
' Web' → 1 token(s):
  [0] 5313 = ' Web'
' NVDA' → 2 token(s):
  [0] 23973 = ' NV'
  [1] 5631 = 'DA'
' screen' → 1 token(s):
  [0] 3159 = ' screen'
' ARIA' → 2 token(s):
  [0] 5923 = ' AR'
  [1] 3539 = 'IA'


tensor([5923, 3539], device='mps:0')

## What Does the Model Predict?
Just let the model complete a prompt. No analysis, just vibes.

In [4]:
def predict(model, prompt, n_tokens=20):
    """Generate a completion. Just talk to the model."""
    tokens = model.to_tokens(prompt)
    for _ in range(n_tokens):
        logits = model(tokens)
        next_token = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        tokens = __import__('torch').cat([tokens, next_token], dim=1)
    completion = model.to_string(tokens[0])
    print(f"Prompt: {prompt}")
    print(f"Completion: {completion}")
    return completion

predict(model, "A screen reader is")

Prompt: A screen reader is
Completion: <|endoftext|>A screen reader is a device that allows you to read text on a screen. It is used to read text on a


'<|endoftext|>A screen reader is a device that allows you to read text on a screen. It is used to read text on a'

## Top-k Predictions
See what the model's top candidates are for the next token.

In [5]:
import torch.nn.functional as F

def top_predictions(model, prompt, k=10):
    """Show the model's top-k next token predictions with probabilities."""
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    probs = F.softmax(logits[0, -1], dim=-1)
    top_probs, top_indices = probs.topk(k)
    
    print(f"Prompt: \"{prompt}\"")
    print(f"\n{'Rank':<6} {'Token':<20} {'Prob':<10}")
    print("-" * 36)
    for i in range(k):
        token_str = model.to_string(top_indices[i])
        print(f"{i+1:<6} '{token_str}'{'':>{18-len(token_str)}} {top_probs[i].item():<10.4f}")

top_predictions(model, "The sky is")

Prompt: "The sky is"

Rank   Token                Prob      
------------------------------------
1      ' the'               0.3355    
2      ' falling'           0.1860    
3      ' blue'              0.0916    
4      ' not'               0.0297    
5      ' a'                 0.0276    
6      ' dark'              0.0189    
7      ' full'              0.0105    
8      ' still'             0.0097    
9      ' now'               0.0092    
10     ' always'            0.0085    


## Quick Logit Lens (single prompt)
When you want a fast look before committing to the full experiment notebook.

In [ ]:
from src.logit_lens import logit_lens, print_logit_lens
from src.viz import plot_logit_lens
import matplotlib.pyplot as plt

# Quick test — change these
prompt = "A screen reader is"
target = " Web"

df, cache = logit_lens(model, prompt, target)
print_logit_lens(df)
plt.close(plot_logit_lens(df))

---
## Scratch Space
Throw anything here. This notebook has no rules.

In [7]:
unload(model)

Model unloaded, memory cleared
